In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import random
import numpy as np

import torch
import torch.nn as nn

import torchvision.datasets as datasets
from torchvision import transforms

from IPython.display import clear_output
from tqdm.notebook import tqdm as tqdm
import ot
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from torch.utils.data import DataLoader, Subset

## Data

In [ ]:
BATCH_SIZE = 16
IMG_SIZE = 16
TRANSFORM = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=TRANSFORM)
idx = np.arange(len(mnist_train))

mnist_2 = Subset(mnist_train, idx[mnist_train.targets == 2])
mnist_3 = Subset(mnist_train, idx[mnist_train.targets == 3])

mnist_2_loader = DataLoader(mnist_2, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
mnist_3_loader = DataLoader(mnist_3, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


def color_batch(imgs, labels):
    # imgs: (B, 1, 16, 16)
    B, _, H, W = imgs.shape
    rgb = torch.zeros(B, 3, H, W, device=imgs.device)

    mask2 = (labels == 2)
    rgb[mask2, 0] = imgs[mask2, 0]

    mask3 = (labels == 3)
    rgb[mask3, 2] = imgs[mask3, 0]

    rgb = (rgb - 0.5) / 0.5

    return rgb

def sample_mnist_2():
    imgs, labels = next(iter(mnist_2_loader))
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    return color_batch(imgs, labels)

def sample_mnist_3():
    imgs, labels = next(iter(mnist_3_loader))
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    return color_batch(imgs, labels)

mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=TRANSFORM)
idx = np.arange(len(mnist_test))

mnist_2_test = Subset(mnist_test, idx[mnist_test.targets == 2])
mnist_3_test = Subset(mnist_test, idx[mnist_test.targets == 3])

mnist_2_test_loader = DataLoader(mnist_2_test, batch_size=BATCH_SIZE)
mnist_3_test_loader = DataLoader(mnist_3_test, batch_size=BATCH_SIZE)

# fixed samples (visualization용)
X_imgs, X_labels = next(iter(mnist_2_test_loader))
Y_imgs, Y_labels = next(iter(mnist_3_test_loader))

X_test_fixed = color_batch(X_imgs.to(DEVICE), X_labels.to(DEVICE))
Y_test_fixed = color_batch(Y_imgs.to(DEVICE), Y_labels.to(DEVICE))


In [ ]:
def plot_images(batch):
    fig, axes = plt.subplots(1, 10, figsize=(10, 1), dpi=100)
    for i in range(10):
        axes[i].imshow(batch[i].cpu().mul(0.5).add(0.5).clip(0,1).permute(1,2,0))
        axes[i].set_xticks([]); axes[i].set_yticks([])
    fig.tight_layout(pad=0.1)

print('Random (unpaired) images from Colored MNIST-2 (1st row) and Colored MNIST-3 (2nd row) train sets')
plot_images(sample_mnist_2())
plot_images(sample_mnist_3())

## Architecture

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c, c, 5, padding=2),
            nn.ReLU(),
            nn.Conv2d(c, c, 5, padding=2)
        )

    def forward(self, x):
        return x + 0.1 * self.block(x)

class TransportNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_in = nn.Conv2d(3, 128, 5, padding=2)

        self.blocks = nn.Sequential(
            ResBlock(128),
            ResBlock(128),
            ResBlock(128),
        )

        self.conv_out = nn.Conv2d(128, 3, 5, padding=2)

    def forward(self, xz):
        x = xz[:, :3]

        h = self.conv_in(xz)
        h = self.blocks(h)
        out = self.conv_out(h)

        return x + 0.1 * out
    
T = TransportNet().to(DEVICE)
f = nn.Sequential(
    nn.Conv2d(3, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  128 x 8 x 8
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  256 x 4 x 4
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  512 x 2 x 2
    nn.Conv2d(128, 128, kernel_size=3, padding=1),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  512 x 1 x 1
    nn.Conv2d(128, 1, kernel_size=1, padding=0),
    nn.Flatten(1),
).to(DEVICE)


print('T params:', np.sum([np.prod(p.shape) for p in T.parameters()]))
print('f params:', np.sum([np.prod(p.shape) for p in f.parameters()]))

def strong_cost(X, T_X):
    X_flat = X.flatten(start_dim=1)
    T_X_flat = T_X.flatten(start_dim=1)

    cost = torch.pow(X_flat - T_X_flat, 2).sum(dim=1).mean()

    return cost


COST = strong_cost


MAX_STEPS = 10000 + 1
T_LR, f_LR = 2e-4, 1e-4
inner_iter = 2
T_opt = torch.optim.Adam(T.parameters(), lr=T_LR, weight_decay=1e-10)
f_opt = torch.optim.Adam(f.parameters(), lr=f_LR, weight_decay=1e-10, maximize = True)

## TTS-EG (Both Semi-Dual Proablem and Lagrangian-based Problem)

In [ ]:
for step in tqdm(range(MAX_STEPS)):

    T_old = [p.clone() for p in T.parameters()]
    f_old = [p.clone() for p in f.parameters()]


    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE) 
    
    T_X = T(X) 

    loss = strong_cost(X, T_X).mean() - f(T_X).mean() + f(Y).mean()
    
    T_opt.zero_grad(); f_opt.zero_grad()
    loss.backward()
    T_opt.step(); f_opt.step()


    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE)
    
    T_X2 = T(X)
    loss2 = strong_cost(X, T_X2).mean() - f(T_X2).mean() + f(Y).mean()

    T_opt.zero_grad(); f_opt.zero_grad()
    loss2.backward()


    with torch.no_grad():
        for p, p_old in zip(T.parameters(), T_old):
            p.copy_(p_old)
        for p, p_old in zip(f.parameters(), f_old):
            p.copy_(p_old)

    T_opt.step(); f_opt.step()


    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()

if f_LR < T_LR:
    name = "Semi-Dual"
elif f_LR > T_LR:
    name = "Lagrangian"
else: # f_LR == T_LR
    name = "Standard"

file_name = name + "_TTS-EG" 

mode_name = "red_to_blue"

os.makedirs(mode_name, exist_ok=True)

save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## GDmax (Semi-Dual Problem)

In [ ]:
for step in tqdm(range(MAX_STEPS)):
    T.train()
    f.train()

    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE)
    
    with torch.no_grad():
        T_X_f = T(X)

    loss_f = - f(T_X_f).mean() + f(Y).mean()
    
    f_opt.zero_grad()
    loss_f.backward()
    f_opt.step()

    for _ in range(inner_iter):
        X = sample_mnist_2().to(DEVICE)
        
        T_X = T(X)
 
        loss_T = strong_cost(X, T_X).mean() - f(T_X).mean()
        
        T_opt.zero_grad()
        loss_T.backward()
        T_opt.step()
    
    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss_T.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()


file_name = "Semi-Dual_GDmax"

mode_name = "red_to_blue"

os.makedirs(mode_name, exist_ok=True)


save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## GDmax (Lagrangian-based problem)

In [ ]:

for step in tqdm(range(MAX_STEPS)):
    T.train()
    f.train()

    X = sample_mnist_2().to(DEVICE)
        
    T_X = T(X)

    loss_T = strong_cost(X, T_X).mean() - f(T_X).mean()
    
    T_opt.zero_grad()
    loss_T.backward()
    T_opt.step()
    
    for _ in range(inner_iter):
        X = sample_mnist_2().to(DEVICE)
        Y = sample_mnist_3().to(DEVICE)
        
        with torch.no_grad():
            T_X_f = T(X)

        loss_f = - f(T_X_f).mean() + f(Y).mean()
        
        f_opt.zero_grad()
        loss_f.backward()
        f_opt.step()

    
    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss_T.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()

file_name = "Lagrangian_GDmax"

mode_name = "red_to_blue"

os.makedirs(mode_name, exist_ok=True)


save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## Evaluation

In [ ]:

T.eval()
all_X_test_colored, all_Y_test_colored = [], []


for imgs, labels in mnist_2_test_loader:
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

    colored_imgs = color_batch(imgs, labels) 
    all_X_test_colored.append(colored_imgs.cpu())

for imgs, labels in mnist_3_test_loader:
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    colored_imgs = color_batch(imgs, labels)
    all_Y_test_colored.append(colored_imgs.cpu())


X_test_all = torch.cat(all_X_test_colored, dim=0).to(DEVICE)
Y_test_all = torch.cat(all_Y_test_colored, dim=0).to(DEVICE)

with torch.no_grad():
    TX_test_all = torch.clamp(T(X_test_all), -1.0, 1.0)

TX_flat = TX_test_all.view(TX_test_all.size(0), -1).cpu().numpy()
X_flat = X_test_all.view(X_test_all.size(0), -1).cpu().numpy()
Y_flat = Y_test_all.view(Y_test_all.size(0), -1).cpu().numpy()


n, m = X_flat.shape[0], Y_flat.shape[0]
a, b = np.ones((n,)) / n, np.ones((m,)) / m
pixel_dim = 3 * 16 * 16 # 768



M_tx_y = ot.dist(TX_flat, Y_flat, metric='sqeuclidean')
w2_tx_y = ot.emd2(a, b, M_tx_y)
per_pixel_w2_tx_y = w2_tx_y / pixel_dim

M_x_y = ot.dist(X_flat, Y_flat, metric='sqeuclidean')
w2_x_y = ot.emd2(a, b, M_x_y)
per_pixel_w2_x_y = w2_x_y / pixel_dim
per_pixel_transport_cost = np.mean(np.sum((X_flat - TX_flat)**2, axis=1)) / pixel_dim
per_pixel_diff = np.abs(per_pixel_w2_x_y - per_pixel_transport_cost)


print("-" * 65)
print(f"{'Metric Item (Per-Pixel)':<40} | {'Value':<15}")
print("-" * 65)
print(f"{'1. D_target':<40} | {per_pixel_w2_tx_y:.6f}")
print(f"{'2. D_cost':<40} | {per_pixel_diff:.6f}")
print("-" * 65)